In [1]:

from pathlib import Path
import random
import numpy as np
import torch

def find_root(start, marker='data/Artemis.csv', max_up=5):
    cur = Path(start).resolve()
    for _ in range(max_up):
        if (cur / marker).exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError(f'{marker} non trouve en remontant depuis {start} ({max_up} niveaux explores)')

ROOT_DIR = find_root(Path.cwd())

DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = ROOT_DIR / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"
RESULTS_DIR = ROOT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"
TABLES_DIR = RESULTS_DIR / "tables"

for directory in [DATA_DIR, PROCESSED_DIR, MODELS_DIR, CHECKPOINTS_DIR, RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DATA_FILE = DATA_DIR / "Artemis.csv"

# --- ce qui manquait : seed et device, utilises par TOUS les notebooks en aval ---
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

print('ROOT_DIR  :', ROOT_DIR)
print('DATA_FILE :', DATA_FILE, '| existe:', DATA_FILE.exists())
print('RANDOM_SEED:', RANDOM_SEED, '| DEVICE:', DEVICE)

ROOT_DIR  : C:\Users\Admin\Desktop\Projet_Artemis2\code
DATA_FILE : C:\Users\Admin\Desktop\Projet_Artemis2\code\data\Artemis.csv | existe: True
RANDOM_SEED: 42 | DEVICE: cuda


In [2]:
POSITIVE_POWER_IS_DISCHARGE = True

EPS_POWER_W = 100.0
SOC_LOW_THRESHOLD = 0.30
HIGH_POWER_THRESHOLD_W = 20000.0

ALPHA_MIN = 0.0
ALPHA_MAX = 1.0

TEMPERATURE_AVAILABLE = False
SOH_AVAILABLE = False

In [3]:
ENERGY_EB_WH = 13709.89
V_EB_PACK_NOM = 450.00
MASS_EB_KG = 55.12

P_EB_MAX_W = 12600.00
P_EB_MIN_W = -6300.00

N_EB_SERIES = 125
N_EB_PARALLEL = 7

SOC_EB_MIN = 0.20
SOC_EB_MAX = 1.00

CAPACITY_EB_AH = ENERGY_EB_WH / V_EB_PACK_NOM

I_EB_MAX_A = P_EB_MAX_W / V_EB_PACK_NOM
I_EB_MIN_A = P_EB_MIN_W / V_EB_PACK_NOM

V_EB_CELL_NOM = V_EB_PACK_NOM / N_EB_SERIES
CAPACITY_EB_CELL_AH = CAPACITY_EB_AH / N_EB_PARALLEL

In [4]:
ENERGY_PB_WH = 2987.12
V_PB_PACK_NOM = 402.60
MASS_PB_KG = 50.02

P_PB_MAX_W = 161040.00
P_PB_MIN_W = -52338.00

N_PB_SERIES = 122
N_PB_PARALLEL = 2

SOC_PB_MIN = 0.20
SOC_PB_MAX = 1.00

CAPACITY_PB_AH = ENERGY_PB_WH / V_PB_PACK_NOM

I_PB_MAX_A = P_PB_MAX_W / V_PB_PACK_NOM
I_PB_MIN_A = P_PB_MIN_W / V_PB_PACK_NOM

V_PB_CELL_NOM = V_PB_PACK_NOM / N_PB_SERIES
CAPACITY_PB_CELL_AH = CAPACITY_PB_AH / N_PB_PARALLEL

In [5]:
ENERGY_HESS_REFERENCE_WH = 16697.02
V_HESS_BUS_NOM = 402.60
MASS_HESS_REFERENCE_KG = 105.14

P_HESS_MAX_REFERENCE_W = 173640.00
P_HESS_MIN_REFERENCE_W = -58638.00

ENERGY_HESS_WH = ENERGY_EB_WH + ENERGY_PB_WH
MASS_HESS_KG = MASS_EB_KG + MASS_PB_KG

P_HESS_MAX_W = P_EB_MAX_W + P_PB_MAX_W
P_HESS_MIN_W = P_EB_MIN_W + P_PB_MIN_W

P_CONV_MAX_W = 1520.00
P_CONV_MIN_W = -760.00

CONVERTER_TYPE = "CCS_PARTIAL_POWER"
CONVERTER_RATIO = None

In [6]:

TIME_COL = "time"

DATA_COLUMNS = [
    "time",
    "speed",
    "hasAeroForce",
    "hasRollingForce",
    "hasGravityForce",
    "hasAcceleration",
    "hasAccelerationForce",
    "hasTotalForce",
    "hasPower",
    "P_EB",
    "P_PB",
    "P_conv",
    "I_EB",
    "I_PB",
    "I_load",
    "I_prime",
    "SOC_EB",
    "SOC_PB",
    "SOC_HESS",
]



In [7]:
WINDOW = 20
HORIZON = 5

DT_SECONDS = None

TRAIN_RATIO = 0.50
VAL_RATIO = 0.25
TEST_RATIO = 0.25

In [8]:
WINDOW = 20
HORIZON = 5

DT_SECONDS = None

TRAIN_RATIO = 0.50
VAL_RATIO = 0.25
TEST_RATIO = 0.25

In [9]:
RANDOM_SEED = 42

LEARNING_RATE = 1e-3
EPOCHS = 100
PATIENCE = 15

NUM_WORKERS = 0

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

PIN_MEMORY = DEVICE.type == "cuda"

In [10]:
LSTM_INPUT_COLS = [
    "speed",
    "hasPower",
    "hasAcceleration",
    "hasTotalForce",
    "SOC_EB",
    "SOC_PB",
    "I_EB",
]

LSTM_OUTPUT_NAMES = [
    "Pdem_future",
    "delta_SOC_EB",
    "delta_SOC_PB",
]

LSTM_HIDDEN_SIZE = 64
LSTM_NUM_LAYERS = 2
LSTM_DROPOUT = 0.20

LSTM_BATCH_SIZE = 64
LSTM_LEARNING_RATE = 1e-3
LSTM_EPOCHS = 100
LSTM_PATIENCE = 15

LSTM_BASELINE_CHECKPOINT = (
    CHECKPOINTS_DIR / "EMS_LSTM.pt"
)

In [11]:
SYMBOLIC_COLS = [
    "EB_available",
    "PB_available",
    "EB_low_SOC",
    "PB_low_SOC",
    "high_power_demand",
    "regenerative_braking",
    "zero_power_demand",
    "converter_risk",
]

In [12]:
LSTM_NS_INPUT_COLS = (
    LSTM_INPUT_COLS
    + SYMBOLIC_COLS
)

LSTM_NS_OUTPUT_NAMES = [
    "Pdem_future",
    "delta_SOC_EB",
    "delta_SOC_PB",
]

LSTM_NS_HIDDEN_SIZE = 64
LSTM_NS_NUM_LAYERS = 2
LSTM_NS_DROPOUT = 0.20

LSTM_NS_BATCH_SIZE = 64
LSTM_NS_LEARNING_RATE = 1e-3
LSTM_NS_EPOCHS = 100
LSTM_NS_PATIENCE = 15

LSTM_NS_LOSS_WEIGHTS = {
    "prediction": 1.00,
    "soc_bounds": 0.10,
    "physics": 0.10,
    "rules": 0.10,
}

LSTM_NS_CHECKPOINT = (
    CHECKPOINTS_DIR / "EMS_LSTM_neurosymbolic.pt"
)

In [13]:
GNN_NODE_NAMES = [
    "energy_battery",
    "power_battery",
    "converter",
    "motor",
    "vehicle",
]

GNN_NODE_FEATURE_NAMES = [
    "soc",
    "current",
    "capacity_ah",
    "power_max_w",
    "power_demand_w",
    "acceleration",
]

GNN_TARGET = "alpha_historical"

GNN_INPUT_DIM = len(GNN_NODE_FEATURE_NAMES)
GNN_HIDDEN_SIZE = 32
GNN_NUM_LAYERS = 2
GNN_DROPOUT = 0.10

GNN_BATCH_SIZE = 64
GNN_LEARNING_RATE = 1e-3
GNN_EPOCHS = 100
GNN_PATIENCE = 15

GNN_CHECKPOINT = (
    CHECKPOINTS_DIR / "EMS_GNN.pt"
)

In [14]:

GNN_NODE_NAMES = [
    "energy_battery",
    "power_battery",
    "converter",
    "motor",
    "vehicle",
]

GNN_CONTINUOUS_FEATURE_NAMES = [
    "soc",
    "current_discharge_max_a",
    "capacity_ah",
    "power_discharge_max_w",
    "power_charge_max_abs_w",
    "power_demand_w",
    "acceleration",
]

GNN_INPUT_DIM = (
    len(GNN_CONTINUOUS_FEATURE_NAMES)
    + len(GNN_NODE_NAMES)
)

GNN_HIDDEN_SIZE = 32
GNN_NUM_LAYERS = 2
GNN_DROPOUT = 0.10

GNN_BATCH_SIZE = 64
GNN_LEARNING_RATE = 1e-3
GNN_EPOCHS = 100
GNN_PATIENCE = 15

GNN_CHECKPOINT = (
    CHECKPOINTS_DIR / "EMS_GNN.pt"
)

GNN_SCALER_FILE = (
    MODELS_DIR / "gnn_node_scalers.npz"
)

GNN_GRAPHS_FILE = (
    PROCESSED_DIR / "hess_graphs.pt"
)



In [15]:
MLP_SIMPLE_INPUT_COLS = [
    "SOC_EB",
    "SOC_PB",
    "hasPower",
    "speed",
    "hasAcceleration",
]

MLP_SIMPLE_TARGET = "alpha_ems_alpha_star"

MLP_SIMPLE_HIDDEN_1 = 64
MLP_SIMPLE_HIDDEN_2 = 32
MLP_SIMPLE_DROPOUT = 0.10

MLP_SIMPLE_BATCH_SIZE = 256
MLP_SIMPLE_LEARNING_RATE = 1e-3
MLP_SIMPLE_EPOCHS = 100
MLP_SIMPLE_PATIENCE = 15

MLP_SIMPLE_CHECKPOINT = (
    CHECKPOINTS_DIR / "EMS_MLP.pt"
)

In [16]:
MLP_NS_INPUT_COLS = [
    "SOC_EB",
    "SOC_PB",
    "hasPower",
    "speed",
    "hasAcceleration",
    "Pdem_pred_ns",
    "delta_soc_eb_pred_ns",
    "delta_soc_pb_pred_ns",
    "alpha_ems_fuzzy_logic",
    "EB_available",
    "PB_available",
    "EB_low_SOC",
    "PB_low_SOC",
    "high_power_demand",
    "regenerative_braking",
    "zero_power_demand",
    "converter_risk",
]

MLP_NS_TARGET = "alpha_ems_alpha_star"

MLP_NS_HIDDEN_1 = 64
MLP_NS_HIDDEN_2 = 32
MLP_NS_DROPOUT = 0.10

MLP_NS_BATCH_SIZE = 256
MLP_NS_LEARNING_RATE = 1e-3
MLP_NS_EPOCHS = 100
MLP_NS_PATIENCE = 15

MAX_DELTA_ALPHA = 0.20
GAMMA_FUSION = 0.30

MLP_NS_LOSS_WEIGHTS = {
    "target": 1.00,
    "rules": 0.10,
    "physics": 0.10,
    "balance": 0.05,
    "continuity": 0.05,
    "converter": 0.05,
}

MLP_NS_CHECKPOINT = (
    CHECKPOINTS_DIR / "EMS_MLP_neurosymbolic.pt"
)

In [17]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [18]:
assert np.isclose(
    TRAIN_RATIO + VAL_RATIO + TEST_RATIO,
    1.0
)

assert np.isclose(
    ENERGY_HESS_WH,
    ENERGY_HESS_REFERENCE_WH,
    atol=0.02
)

assert np.isclose(
    MASS_HESS_KG,
    MASS_HESS_REFERENCE_KG
)

assert np.isclose(
    P_HESS_MAX_W,
    P_HESS_MAX_REFERENCE_W
)

assert np.isclose(
    P_HESS_MIN_W,
    P_HESS_MIN_REFERENCE_W
)

assert 0 <= SOC_EB_MIN < SOC_EB_MAX <= 1
assert 0 <= SOC_PB_MIN < SOC_PB_MAX <= 1

assert P_EB_MIN_W <= 0 <= P_EB_MAX_W
assert P_PB_MIN_W <= 0 <= P_PB_MAX_W
assert P_CONV_MIN_W <= 0 <= P_CONV_MAX_W

assert WINDOW > 0
assert HORIZON > 0

assert 0 < MAX_DELTA_ALPHA <= 1
assert 0 <= GAMMA_FUSION <= 1

assert MLP_SIMPLE_TARGET == "alpha_ems_alpha_star"
assert MLP_NS_TARGET == "alpha_ems_alpha_star"

assert all(
    value >= 0
    for value in LSTM_NS_LOSS_WEIGHTS.values()
)

assert all(
    value >= 0
    for value in MLP_NS_LOSS_WEIGHTS.values()
)

In [19]:

print("CONFIGURATION DU PROJET HESS")


print("\nBATTERIE ÉNERGIE")
print(f"Énergie          : {ENERGY_EB_WH:.2f} Wh")
print(f"Tension          : {V_EB_PACK_NOM:.2f} V")
print(f"Capacité         : {CAPACITY_EB_AH:.4f} Ah")
print(f"Masse            : {MASS_EB_KG:.2f} kg")
print(f"Courant recharge : {I_EB_MIN_A:.2f} A")
print(f"Courant décharge : {I_EB_MAX_A:.2f} A")
print(f"Configuration    : {N_EB_SERIES}S{N_EB_PARALLEL}P")

print("\nBATTERIE PUISSANCE")
print(f"Énergie          : {ENERGY_PB_WH:.2f} Wh")
print(f"Tension          : {V_PB_PACK_NOM:.2f} V")
print(f"Capacité         : {CAPACITY_PB_AH:.4f} Ah")
print(f"Masse            : {MASS_PB_KG:.2f} kg")
print(f"Courant recharge : {I_PB_MIN_A:.2f} A")
print(f"Courant décharge : {I_PB_MAX_A:.2f} A")
print(f"Configuration    : {N_PB_SERIES}S{N_PB_PARALLEL}P")

print("\nHESS")
print(f"Énergie totale   : {ENERGY_HESS_WH:.2f} Wh")
print(f"Masse totale     : {MASS_HESS_KG:.2f} kg")
print(f"Tension du bus   : {V_HESS_BUS_NOM:.2f} V")
print(f"Puissance min    : {P_HESS_MIN_W:.2f} W")
print(f"Puissance max    : {P_HESS_MAX_W:.2f} W")

print("\nMODÈLES")
print(
    f"LSTM seul        : "
    f"{len(LSTM_INPUT_COLS)} → "
    f"{LSTM_HIDDEN_SIZE} → 3"
)

print(
    f"LSTM NS          : "
    f"{len(LSTM_NS_INPUT_COLS)} → "
    f"{LSTM_NS_HIDDEN_SIZE} → 3"
)

print(
    f"GNN simple       : "
    f"{GNN_INPUT_DIM} → "
    f"{GNN_HIDDEN_SIZE} → 1"
)

print(
    f"MLP simple       : "
    f"{len(MLP_SIMPLE_INPUT_COLS)} → "
    f"{MLP_SIMPLE_HIDDEN_1} → "
    f"{MLP_SIMPLE_HIDDEN_2} → 1"
)

print(
    f"MLP NS           : "
    f"{len(MLP_NS_INPUT_COLS)} → "
    f"{MLP_NS_HIDDEN_1} → "
    f"{MLP_NS_HIDDEN_2} → 1"
)

print(f"\nDevice           : {DEVICE}")
print(f"Fichier          : {DATA_FILE}")
print("\nConfiguration validée.")

CONFIGURATION DU PROJET HESS

BATTERIE ÉNERGIE
Énergie          : 13709.89 Wh
Tension          : 450.00 V
Capacité         : 30.4664 Ah
Masse            : 55.12 kg
Courant recharge : -14.00 A
Courant décharge : 28.00 A
Configuration    : 125S7P

BATTERIE PUISSANCE
Énergie          : 2987.12 Wh
Tension          : 402.60 V
Capacité         : 7.4196 Ah
Masse            : 50.02 kg
Courant recharge : -130.00 A
Courant décharge : 400.00 A
Configuration    : 122S2P

HESS
Énergie totale   : 16697.01 Wh
Masse totale     : 105.14 kg
Tension du bus   : 402.60 V
Puissance min    : -58638.00 W
Puissance max    : 173640.00 W

MODÈLES
LSTM seul        : 7 → 64 → 3
LSTM NS          : 15 → 64 → 3
GNN simple       : 12 → 32 → 1
MLP simple       : 5 → 64 → 32 → 1
MLP NS           : 17 → 64 → 32 → 1

Device           : cuda
Fichier          : C:\Users\Admin\Desktop\Projet_Artemis2\code\data\Artemis.csv

Configuration validée.
